In [1]:
# %pip install python-dotenv
from dotenv import load_dotenv

load_dotenv()

True

# YouTube Chatbot with RAG using LangChain

This notebook implements a Retrieval-Augmented Generation (RAG) system that allows users to ask questions about YouTube video transcripts. The system fetches the transcript, processes it into chunks, embeds them using Google Generative AI, stores them in a FAISS vector database, and uses LangChain to retrieve relevant context and generate answers.

## Overview of the Process:
1. **Environment Setup**: Load environment variables and install dependencies.
2. **Transcript Fetching**: Retrieve the transcript from a YouTube video.
3. **Text Splitting**: Break the transcript into manageable chunks.
4. **Embedding and Storage**: Create embeddings and store in FAISS vector store.
5. **Retrieval Setup**: Configure a retriever for similarity search.
6. **Chain Building**: Build a LangChain pipeline for question answering.
7. **Querying**: Ask questions and get answers based on the transcript.

In [2]:
%pip install -q youtube-transcript-api langchain-community google-generativeai faiss-cpu tiktoken langchain langchain_text_splitters langchain-google-genai langchain-core

Note: you may need to restart the kernel to use updated packages.


In [3]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 2: Fetch YouTube Transcript

In this section, we fetch the transcript of a YouTube video using the YouTube Transcript API. We specify the video ID and attempt to get the transcript in English or Hindi. If transcripts are disabled, an error is raised.

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi

print(dir(YouTubeTranscriptApi))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'fetch', 'list']


In [8]:
video_id = "etnLX7m2MiA"
try:
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id,languages=["en", "hi"])
    transcript = " ".join([item.text for item in transcript_list])
except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")

In [9]:
print(transcript[:500])

हाय गाइस, माय नेम इज नितेश एंड यू वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू करेंगे। अब वीडियो को स्टार्ट करने के पहले एक बार मैं आपको क्विक रिककैप देना चाहूंगा कि अभी तक हमने इस प्लेलिस्ट में क्या-क्या कवर किया है। इस प्लेलिस्ट में अभी तक मैंने 15 वीडियोस डाले हैं और अभी तक के 15 वीडियोस को हम दो पार्ट्स में डिवाइड कर सकते हैं। इन द फर्स्ट पार्ट, हमने लैंग चेन के फंडामेंटल्स कवर किए। जहां पर मैंने आपको यह बताया कि लैंग चेन में कौन-कौन से कंपोनेंट्स होते 


## Step 3: Text Splitting

To handle large transcripts effectively, we split the text into smaller chunks. This improves retrieval accuracy and manages token limits. We use RecursiveCharacterTextSplitter with a chunk size of 1000 characters and an overlap of 200 characters.

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_text(transcript)

In [11]:
len(chunks)

48

In [13]:
chunks[0]

'हाय गाइस, माय नेम इज नितेश एंड यू वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू करेंगे। अब वीडियो को स्टार्ट करने के पहले एक बार मैं आपको क्विक रिककैप देना चाहूंगा कि अभी तक हमने इस प्लेलिस्ट में क्या-क्या कवर किया है। इस प्लेलिस्ट में अभी तक मैंने 15 वीडियोस डाले हैं और अभी तक के 15 वीडियोस को हम दो पार्ट्स में डिवाइड कर सकते हैं। इन द फर्स्ट पार्ट, हमने लैंग चेन के फंडामेंटल्स कवर किए। जहां पर मैंने आपको यह बताया कि लैंग चेन में कौन-कौन से कंपोनेंट्स होते हैं। और फिर एक-एक करके हमने उन कंपोनेंट्स को डिस्कस किया। जैसे कि मॉडल्स वाला कंपोनेंट, प्र्प्ट्स वाला कंपोनेंट, चेस वाला कंपोनेंट एक्सट्रा। फिर जो सेकंड पार्ट था वहां पे हमने लैंग चेन को यूज़ करके रैक सिस्टम्स बनाना सीखा। सो वहां पर हमने टॉपिक्स कवर किए जैसे कि डॉक्यूमेंट लोडर्स, टेक्स्ट स्प्लिटर्स, वेक्टर स्टोर्स, रिट्रीवर्स और फाइनली हमने रैक बेस सिस्टम बनाना सीखा। अब यहां से आज के वीडियो से हम लोग इस प्लेलिस्ट का थर्ड पार्ट शुरू करने जा रहे हैं। जहां पर हम लैंग चेन को यूज करके एजेंट्स कैसे'

## Step 4: Embeddings and Vector Store

We create embeddings for each text chunk using Google Generative AI's embedding model. These embeddings are then stored in a FAISS vector database for efficient similarity search.

In [14]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)
vectorstore = FAISS.from_texts(chunks, embeddings)

In [15]:
vectorstore.index_to_docstore_id

{0: 'f7cdb1e2-7262-441a-8b64-b57969eceb0b',
 1: 'bc467182-318d-4fcd-8b6e-0d37d9e871b9',
 2: '7f433bee-fc7e-41b7-82b8-d8a92f0b7011',
 3: 'ea10c735-70fc-48c1-8aea-378f0c823f19',
 4: '8c23ab8d-e8e2-438e-9e6b-b117d7ddd948',
 5: 'ec2af3a5-a4d6-43c6-905c-383273d1ffda',
 6: 'ed545b6d-2c28-486b-869a-26c26795357b',
 7: 'e60160a0-2193-4fe5-b784-872150b2cde2',
 8: 'bd3052f8-44ff-4990-86ef-4e76b4f7d2fe',
 9: 'b9d2f5dd-f99a-4d2d-ba5f-9772a79fae24',
 10: '78044c7a-02ee-429d-a1a2-118262155524',
 11: 'c0235560-ff50-4bb3-a55a-30bc15f7fcd3',
 12: '8a2fade4-cbd8-4949-b1fd-73a1ed302f7d',
 13: 'e9b65753-020d-48d6-af13-f76e28b1a598',
 14: 'bc3d79c9-849b-4b5e-85bc-0efbf08500a3',
 15: 'b2ad2e5e-bbd9-44f9-8cfc-631400b93258',
 16: 'e94175b3-acfe-4f54-8ea7-c94579bd64c6',
 17: '36a781fd-18f2-4f00-b641-c3e0b7cd36e3',
 18: '63e4b8a0-0753-49e7-8f03-cfd2036c989a',
 19: '562fe27e-9611-48c5-90bf-0af8358fb045',
 20: '6c3bdad7-da02-4edf-b384-0dca11c300f0',
 21: 'ba693c81-cddf-49df-9383-8905649a4d6e',
 22: '79e7100b-afd2-

In [16]:
vectorstore.get_by_ids(["1968bcdb-01ef-44db-a043-d3fdfc2e322c"])

[Document(id='1968bcdb-01ef-44db-a043-d3fdfc2e322c', metadata={}, page_content='टूलकिट्स कि आप सबको साथ में एक्सेस भी कर पा रहे हो और साथ ही साथ रीयूजेबिलिटी का पॉइंट ऑफ व्यू तो है ही। ठीक है? सो, बहुत ही छोटा सा कांसेप्ट था। बट जस्ट मुझे लगा कि कवर कर देना चाहिए। सो या अह दैट्स इट फॉर द वीडियो, गाइज़। आई नो आपको थोड़ा सा ऐसा अधूरा-अधूरा सा लग रहा होगा कि हां यार हमने टूल्स बनाना सीख तो लिया बट हमने एलएलएम के साथ कनेक्ट कैसे करना है इन टूल्स को ये तो सीखा ही नहीं। उसका रीज़न यह है कि ये टॉपिक हम लोग नेक्स्ट वीडियो में कवर करेंगे जब मैं आपको टूल कॉलिंग पढ़ाऊंगा। अ सो आज के वीडियो का मेन फंडा था आपको टूल्स के बारे में पूरा ज्ञान देना। आई रियली होप वो मैं दे पाया। आपको टूल्स के अराउंड सब कुछ क्लियर हो चुका है। अब अगला जो वीडियो है इनफैक्ट अगला और उसके अगला वीडियो दोनों बहुत इंटरेस्टिंग है। बिकॉज़ आगे की वीडियोस में हम इन टूल्स को यूज करेंगे एलएलएम्स के साथ। ठीक है? सो अगर आपको यह वीडियो पसंद आया प्लीज लाइक करना। अगर आपने इस चैनल को सब्सक्राइब नहीं किया है, प्लीज डू सब्सक्राइब। मिलते हैं नेक्

## Step 5: Retrieval Setup

We set up a retriever that performs similarity search on the vector store, returning the top 4 most relevant chunks for a given query.

In [17]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [18]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x114c37110>, search_kwargs={'k': 4})

In [19]:
retriever.invoke("What is the video about?")

[Document(id='7f433bee-fc7e-41b7-82b8-d8a92f0b7011', metadata={}, page_content='तो उस सेंस में आज का वीडियो इज़ सुपरेंट। सो प्लीज मेक श्योर आप इस वीडियो को एंड टू एंड देखो। नाउ लेट्स स्टार्ट द वीडियो। अब वीडियो में आगे बढ़ने के पहले मैं आपको एक ओवरव्यू देना चाहता हूं कि ये जो सेगमेंट है हमारा एजेंट्स वाला इसमें हम एग्जैक्टली कौन-कौन से टॉपिक्स कवर करने वाले हैं। तो सबसे पहले हम लोग कवर करेंगे टूल्स जो आज के वीडियो का टॉपिक है। यहां पे हम अलग-अलग टाइप के टूल्स के साथ काम करना सीखेंगे। खुद के टूल्स भी बनाना सीखेंगे। अ उसके बाद जो अगला वीडियो है उसमें हम एक कांसेप्ट कवर करेंगे जिसको बोला जाता है टूल कॉलिंग। सो टूल कॉलिंग में आप क्या करते हो कि आपने जो टूल बनाया है और आप अपने एलएलएम को एक दूसरे के साथ कनेक्ट करते हो ताकि ये दोनों साथ में काम कर पाएं। ठीक है? तो ये हमारा नेक्स्ट वीडियो का टॉपिक है। उसके बाद हम कवर करेंगे एजेंट्स का कांसेप्ट। ठीक है? जहां पर हम लैंग चेन को यूज़ करके एजेंट्स बनाएंगे। और आप नोटिस करोगे कि इन एजेंट्स को बनाने के प्रोसेस में आप एलएलएम्स को यूज़ करोगे, टूल्स को यूज़ 

## Step 6: Language Model and Prompt Setup

We initialize the Google Generative AI model (Gemini 3 Flash) and define a prompt template that instructs the model to answer questions based only on the provided context from the transcript.

In [20]:
llm = ChatGoogleGenerativeAI(model = "gemini-3-flash-preview",temperature=0.7)

In [22]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer ONLY from the provided transscript context.
    If the context is insuffiecent , just say you don't know.
    
    {context}
    Question: {question}
    """,
    optional_variables=["context", "question"]
)

In [24]:
question = "What is the video about?"
retrieved_docs = retriever.invoke(question)

In [36]:
retrieved_docs

[Document(id='7f433bee-fc7e-41b7-82b8-d8a92f0b7011', metadata={}, page_content='तो उस सेंस में आज का वीडियो इज़ सुपरेंट। सो प्लीज मेक श्योर आप इस वीडियो को एंड टू एंड देखो। नाउ लेट्स स्टार्ट द वीडियो। अब वीडियो में आगे बढ़ने के पहले मैं आपको एक ओवरव्यू देना चाहता हूं कि ये जो सेगमेंट है हमारा एजेंट्स वाला इसमें हम एग्जैक्टली कौन-कौन से टॉपिक्स कवर करने वाले हैं। तो सबसे पहले हम लोग कवर करेंगे टूल्स जो आज के वीडियो का टॉपिक है। यहां पे हम अलग-अलग टाइप के टूल्स के साथ काम करना सीखेंगे। खुद के टूल्स भी बनाना सीखेंगे। अ उसके बाद जो अगला वीडियो है उसमें हम एक कांसेप्ट कवर करेंगे जिसको बोला जाता है टूल कॉलिंग। सो टूल कॉलिंग में आप क्या करते हो कि आपने जो टूल बनाया है और आप अपने एलएलएम को एक दूसरे के साथ कनेक्ट करते हो ताकि ये दोनों साथ में काम कर पाएं। ठीक है? तो ये हमारा नेक्स्ट वीडियो का टॉपिक है। उसके बाद हम कवर करेंगे एजेंट्स का कांसेप्ट। ठीक है? जहां पर हम लैंग चेन को यूज़ करके एजेंट्स बनाएंगे। और आप नोटिस करोगे कि इन एजेंट्स को बनाने के प्रोसेस में आप एलएलएम्स को यूज़ करोगे, टूल्स को यूज़ 

In [25]:
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [37]:
context_text

'तो उस सेंस में आज का वीडियो इज़ सुपरेंट। सो प्लीज मेक श्योर आप इस वीडियो को एंड टू एंड देखो। नाउ लेट्स स्टार्ट द वीडियो। अब वीडियो में आगे बढ़ने के पहले मैं आपको एक ओवरव्यू देना चाहता हूं कि ये जो सेगमेंट है हमारा एजेंट्स वाला इसमें हम एग्जैक्टली कौन-कौन से टॉपिक्स कवर करने वाले हैं। तो सबसे पहले हम लोग कवर करेंगे टूल्स जो आज के वीडियो का टॉपिक है। यहां पे हम अलग-अलग टाइप के टूल्स के साथ काम करना सीखेंगे। खुद के टूल्स भी बनाना सीखेंगे। अ उसके बाद जो अगला वीडियो है उसमें हम एक कांसेप्ट कवर करेंगे जिसको बोला जाता है टूल कॉलिंग। सो टूल कॉलिंग में आप क्या करते हो कि आपने जो टूल बनाया है और आप अपने एलएलएम को एक दूसरे के साथ कनेक्ट करते हो ताकि ये दोनों साथ में काम कर पाएं। ठीक है? तो ये हमारा नेक्स्ट वीडियो का टॉपिक है। उसके बाद हम कवर करेंगे एजेंट्स का कांसेप्ट। ठीक है? जहां पर हम लैंग चेन को यूज़ करके एजेंट्स बनाएंगे। और आप नोटिस करोगे कि इन एजेंट्स को बनाने के प्रोसेस में आप एलएलएम्स को यूज़ करोगे, टूल्स को यूज़ करोगे, टूल कॉलिंग के कांसेप्ट को यूज़ करोगे। तो बेसिकली अभी तक आपने जो भी पढ़ा\n

In [26]:
final_prompt = prompt.format(context=context_text, question=question)

In [38]:
final_prompt

"\n    You are a helpful assistant.\n    Answer ONLY from the provided transscript context.\n    If the context is insuffiecent , just say you don't know.\n\n    तो उस सेंस में आज का वीडियो इज़ सुपरेंट। सो प्लीज मेक श्योर आप इस वीडियो को एंड टू एंड देखो। नाउ लेट्स स्टार्ट द वीडियो। अब वीडियो में आगे बढ़ने के पहले मैं आपको एक ओवरव्यू देना चाहता हूं कि ये जो सेगमेंट है हमारा एजेंट्स वाला इसमें हम एग्जैक्टली कौन-कौन से टॉपिक्स कवर करने वाले हैं। तो सबसे पहले हम लोग कवर करेंगे टूल्स जो आज के वीडियो का टॉपिक है। यहां पे हम अलग-अलग टाइप के टूल्स के साथ काम करना सीखेंगे। खुद के टूल्स भी बनाना सीखेंगे। अ उसके बाद जो अगला वीडियो है उसमें हम एक कांसेप्ट कवर करेंगे जिसको बोला जाता है टूल कॉलिंग। सो टूल कॉलिंग में आप क्या करते हो कि आपने जो टूल बनाया है और आप अपने एलएलएम को एक दूसरे के साथ कनेक्ट करते हो ताकि ये दोनों साथ में काम कर पाएं। ठीक है? तो ये हमारा नेक्स्ट वीडियो का टॉपिक है। उसके बाद हम कवर करेंगे एजेंट्स का कांसेप्ट। ठीक है? जहां पर हम लैंग चेन को यूज़ करके एजेंट्स बनाएंगे। और आप नोटिस क

In [28]:
answer = llm.invoke(final_prompt)
print(answer.content)

[{'type': 'text', 'text': 'Based on the transcript provided, the video is about **Tools (टूल्स)** in the context of LangChain. \n\nSpecifically:\n* It marks the beginning of the **third part** of a LangChain playlist, which focuses on building **Agents**.\n* The video covers how to work with different types of tools and how to **create your own tools**.\n* It serves as a foundation for upcoming topics like "Tool Calling" and building "Agents."', 'extras': {'signature': 'EtgOCtUOAb4+9vt/6VdGzPMZjWRNWVVXrMhH7+1OdtanNLaPD8fdUItRcdJA/RvdL+SredCnAVgl/cTxulzEO1whn2KNEoKye3jI5ZquF2gvmxHNCZFYIbWGPzOweQRROto3O5+s52YEfbroRc5vwgqUQQtYr9J2DNNtOev4BXCIT2y+TYosgTfJSSg/ygLIOmF539EA35b8BUmPy2+nJhbS40K6xKHXbAZGtPHjdHCLydwCaG1Q0SQff94CHlVM+mX/Ia7JkYDpsAeMAleJRomeZg0xeO8Xint1isPZGzH/1vMoTFB5cRXB9fN1blR0YPd3+ztnHLZtR1L+soscCLkgMOX5xCzphMY/ciHys9Xt4XgZCFNS0ap1qEB+ATyCx6OS+nlCSBEgczPheoCV97dQjM+ehMRE7Ec+8MDUhU9qQJk/cgDRClkWR2BVqPusnvO9XnkweL1hEiSy8gkJxh+doW9IERjYLnM4KrE7As3WZhUaeNrOa4zFTvD9jhYWS4Bl9bp7TNlk2

## Step 7: Building the RAG Chain

We construct a LangChain pipeline that combines retrieval, context formatting, prompting, LLM inference, and output parsing into a single runnable chain.

In [29]:
from langchain_core.runnables import RunnableLambda,RunnableParallel,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
    """
    Formats the retrieved documents into a single string separated by double newlines.

    This function takes a list of retrieved documents (typically from a vector store)
    and concatenates their page_content attributes into a single string, with each
    document's content separated by two newline characters for readability.

    Args:
        retrieved_docs (list): A list of document objects, each with a 'page_content' attribute.

    Returns:
        str: A single string containing the concatenated page content of all documents.
    """
    return "\n\n".join([doc.page_content for doc in retrieved_docs])

In [31]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})


In [32]:
parallel_chain.invoke("What is the video about?")

{'context': 'तो उस सेंस में आज का वीडियो इज़ सुपरेंट। सो प्लीज मेक श्योर आप इस वीडियो को एंड टू एंड देखो। नाउ लेट्स स्टार्ट द वीडियो। अब वीडियो में आगे बढ़ने के पहले मैं आपको एक ओवरव्यू देना चाहता हूं कि ये जो सेगमेंट है हमारा एजेंट्स वाला इसमें हम एग्जैक्टली कौन-कौन से टॉपिक्स कवर करने वाले हैं। तो सबसे पहले हम लोग कवर करेंगे टूल्स जो आज के वीडियो का टॉपिक है। यहां पे हम अलग-अलग टाइप के टूल्स के साथ काम करना सीखेंगे। खुद के टूल्स भी बनाना सीखेंगे। अ उसके बाद जो अगला वीडियो है उसमें हम एक कांसेप्ट कवर करेंगे जिसको बोला जाता है टूल कॉलिंग। सो टूल कॉलिंग में आप क्या करते हो कि आपने जो टूल बनाया है और आप अपने एलएलएम को एक दूसरे के साथ कनेक्ट करते हो ताकि ये दोनों साथ में काम कर पाएं। ठीक है? तो ये हमारा नेक्स्ट वीडियो का टॉपिक है। उसके बाद हम कवर करेंगे एजेंट्स का कांसेप्ट। ठीक है? जहां पर हम लैंग चेन को यूज़ करके एजेंट्स बनाएंगे। और आप नोटिस करोगे कि इन एजेंट्स को बनाने के प्रोसेस में आप एलएलएम्स को यूज़ करोगे, टूल्स को यूज़ करोगे, टूल कॉलिंग के कांसेप्ट को यूज़ करोगे। तो बेसिकली अभी तक आपने 

In [33]:
parser = StrOutputParser()

In [34]:
main_chain = parallel_chain | prompt | llm | parser

In [39]:
main_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x114c37110>, search_kwargs={'k': 4})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| PromptTemplate(input_variables=['context', 'question'], optional_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n    You are a helpful assistant.\n    Answer ONLY from the provided transscript context.\n    If the context is insuffiecent , just say you don't know.\n\n    {context}\n    Question: {question}\n    ")
| ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_i

In [35]:
main_chain.invoke("What is the video about?")

'Based on the transcript provided, the video is about the following:\n\n*   **LangChain Playlist:** It is a continuation of a LangChain playlist, marking the start of the "third part" of the series.\n*   **Tools:** The primary focus of this specific video is "Tools" (टूल्स). It covers working with different types of tools and learning how to create your own tools.\n*   **Introduction to Agents:** It serves as an introduction to the broader segment on "Agents," explaining how tools, tool calling, and LLMs work together to build agents. \n*   **Knowledge Overview:** The goal of the video is to provide complete knowledge about tools before moving on to "Tool Calling" and "Agents" in subsequent videos.'

## Usage

To use this chatbot:

1. Change the `video_id` variable to the ID of your desired YouTube video.
2. Run all cells up to the chain creation.
3. Invoke the `main_chain` with your question: `main_chain.invoke("Your question here?")`

### Example Queries:
- "What is the video about?"
- "Summarize the main points."
- "What are the key takeaways?"

Note: The system answers based only on the video transcript. If the context is insufficient, it will state that it doesn't know.